In [ ]:
import sys
import os
import joblib
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model

# 1. Chỉ đường cho Python ra thư mục gốc TTCS
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

# 2. Import các hàm từ src của nhóm mình
from src.utils import evaluate_model
from src.data_loader import load_and_preprocess

print("--- Đang nạp mô hình và dữ liệu CNN ---")

# 3. Tải lại mô hình và Label Encoder (Nhớ dấu ../)
model = load_model('../models/best_cnn_model.h5')
le = joblib.load('../models/label_encoder_cnn.pkl')

# 4. Tải dữ liệu Test để đánh giá
_, X_test_s, _, y_test, _, _ = load_and_preprocess('../data/Dataset-Unicauca-Version2-87Atts.csv')
X_test_3d = np.expand_dims(X_test_s, axis=2)

# 5. Dự đoán và Xử lý nhãn
print("--- Đang xử lý dự đoán... ---")
y_pred = model.predict(X_test_3d)
y_pred_classes = np.argmax(y_pred, axis=1)

# Lọc bỏ nhãn lạ (ví dụ nhãn 39)
valid_mask = np.isin(y_test, le.classes_)
y_test_filtered = y_test[valid_mask]
y_pred_filtered = y_pred_classes[valid_mask]

# Chuyển đổi về dạng số
y_test_fixed = le.transform(y_test_filtered)

# Lấy danh sách các nhãn thực tế xuất hiện (dạng số)
actual_labels = np.unique(y_test_fixed)

# ÉP KIỂU VỀ STRING: Đảm bảo actual_names là danh sách các chuỗi tên App
actual_names = [str(le.classes_[i]) for i in actual_labels]

print("\n--- KẾT QUẢ F1-SCORE---")
from sklearn.metrics import classification_report

# Chạy lệnh này là bảng hiện ra ngay!
report = classification_report(y_test_fixed, y_pred_filtered, labels=actual_labels, target_names=actual_names)
print(report)

--- Đang nạp mô hình và dữ liệu... ---
--- Đang xử lý dự đoán... ---
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 21s 11ms/step

--- KẾT QUẢ F1-SCORE---
              precision    recall  f1-score   support

           0       0.75      0.46      0.57      1400
           1       0.36      0.43      0.39       139
           2       0.05      0.26      0.09        23
           3       0.03      0.16      0.05        19
           7       0.00      0.00      0.00         1
           8       0.75      0.41      0.53       264
           9       0.86      0.86      0.86       145
          10       0.00      0.00      0.00         1
          11       0.45      0.95      0.62        21
          12       0.87      0.72      0.79       431
          13       0.02      0.43      0.03         7
          14       0.02      0.14      0.04        22
          16       0.88      0.72      0.79       485
          17       0.00      0.00      0.00         1
          18       0.00      0.00      0.00       

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [7]:
import sys
import os
import joblib
import numpy as np
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report

# 1. Chỉ đường cho Python ra thư mục gốc TTCS để tìm folder src
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data_loader import load_and_preprocess

print("--- Đang nạp mô hình và dữ liệu LSTM ---")

# 2. Tải mô hình LSTM và Label Encoder tương ứng
# Đảm bảo Mạnh đã chạy train_lstm.py để có file .pkl này nhé
model_lstm = load_model('../models/best_lstm_model.h5')

# Kiểm tra xem có file label_encoder_lstm.pkl không, nếu không thì dùng tạm của cnn
if os.path.exists('../models/label_encoder_lstm.pkl'):
    le = joblib.load('../models/label_encoder_lstm.pkl')
    print("Sử dụng: label_encoder_lstm.pkl")
else:
    le = joblib.load('../models/label_encoder_cnn.pkl')
    print("Sử dụng: label_encoder_cnn.pkl (Cảnh báo: Có thể lệch nhãn nếu train khác tập)")

# 3. Tải dữ liệu Test
_, X_test_s, _, y_test, _, _ = load_and_preprocess('../data/Dataset-Unicauca-Version2-87Atts.csv')

# 4. Reshape dữ liệu về dạng 3D cho LSTM
# axis=2 tạo ra (Samples, Features, 1) - khớp với cấu trúc em train
X_test_3d = np.expand_dims(X_test_s, axis=2)

print("--- Đang xử lý dự đoán LSTM... ---")
y_pred = model_lstm.predict(X_test_3d)
y_pred_classes = np.argmax(y_pred, axis=1)

# 5. Xử lý lỗi nhãn lạ (nhãn 39) để không bị crash
valid_mask = np.isin(y_test, le.classes_)
y_test_filtered = y_test[valid_mask]
y_pred_filtered = y_pred_classes[valid_mask]

# Chuyển đổi nhãn về dạng số chuẩn
y_test_fixed = le.transform(y_test_filtered)

# Lấy danh sách tên các App thực tế có trong tập test để in bảng
actual_labels = np.unique(y_test_fixed) 
actual_names = [str(le.classes_[i]) for i in actual_labels]

print("\n--- KẾT QUẢ F1-SCORE LSTM ---")
report = classification_report(y_test_fixed, y_pred_filtered, labels=actual_labels, target_names=actual_names)
print(report)

--- Đang nạp mô hình và dữ liệu LSTM ---


Sử dụng: label_encoder_lstm.pkl
--- Đang xử lý dự đoán LSTM... ---
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 79s 42ms/step

--- KẾT QUẢ F1-SCORE LSTM ---
              precision    recall  f1-score   support

           0       0.80      0.36      0.50      1400
           1       0.28      0.33      0.30       139
           2       0.03      0.17      0.05        23
           3       0.01      0.05      0.02        19
           7       0.00      0.00      0.00         1
           8       0.77      0.34      0.47       264
           9       0.83      0.84      0.84       145
          10       0.00      0.00      0.00         1
          11       0.49      0.90      0.63        21
          12       0.89      0.67      0.77       431
          13       0.08      0.43      0.14         7
          14       0.03      0.09      0.04        22
          16       0.90      0.67      0.77       485
          17       0.00      0.00      0.00         1
          18       0.00      0.00      0.00   

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [ ]:
import sys
import os
import joblib
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# 1. Chỉ đường cho Python ra thư mục gốc TTCS
root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from src.data_loader import load_and_preprocess

print("--- Đang nạp mô hình và dữ liệu XGBoost ---")

# 2. Tải mô hình XGBoost và các bộ mã hóa nhãn
# Mình cần cả LabelEncoder gốc (le) và cái Encoder nội bộ của XGBoost (le_xg)
model_xgb = XGBClassifier()
model_xgb.load_model('../models/best_xgboost_model.json')

le_final = joblib.load('../models/label_encoder_xgboost.pkl') # Encoder gốc (tên App)
le_xg = joblib.load('../models/label_encoder_xgboost_internal.pkl') # Encoder phụ (0, 1, 2...)

# 3. Tải dữ liệu Test
_, X_test_s, _, y_test, _, _ = load_and_preprocess('../data/Dataset-Unicauca-Version2-87Atts.csv')

# XGBoost dùng dữ liệu 2D (không cần expand_dims như CNN/LSTM)
X_test_2d = X_test_s.reshape(X_test_s.shape[0], -1)

print("--- Đang xử lý dự đoán XGBoost... ---")
# Dự đoán ra các nhãn 0, 1, 2...
y_pred_encoded = model_xgb.predict(X_test_2d)

# 4. LỌC KÉP: Chỉ lấy những nhãn mà CẢ LabelEncoder gốc VÀ Encoder của XGBoost đều biết
# Bước A: Lọc theo le_final (nhãn 39 và các nhãn không có trong bộ nhớ gốc)
valid_mask_final = np.isin(y_test, le_final.classes_)

# Bước B: Lọc theo le_xg (những nhãn mà XGBoost thực sự được học lúc train)
valid_mask_xg = np.isin(y_test, le_xg.classes_)

# Kết hợp cả 2 điều kiện
final_mask = valid_mask_final & valid_mask_xg

y_test_filtered = y_test[final_mask]
y_pred_filtered = y_pred_encoded[final_mask]

# 5. Bây giờ transform chắc chắn không lỗi vì đã lọc hết "người lạ"
y_test_encoded = le_xg.transform(y_test_filtered)

# 6. Lấy tên App chuẩn
actual_class_indices = le_xg.classes_ 
actual_names = [str(le_final.classes_[i]) for i in actual_class_indices]

print("\n--- KẾT QUẢ F1-SCORE XGBOOST ---")
from sklearn.metrics import classification_report
# Quan trọng: labels=np.arange(len(actual_names)) để khớp với số lớp XGBoost học
report = classification_report(y_test_encoded, y_pred_filtered, 
                               labels=np.arange(len(actual_names)),
                               target_names=actual_names)
print(report)

--- Đang nạp mô hình và dữ liệu XGBoost ---
--- Đang xử lý dự đoán XGBoost... ---

--- KẾT QUẢ F1-SCORE XGBOOST CHO HIẾU ---
              precision    recall  f1-score   support

           0       0.82      0.49      0.62      1400
           1       0.46      0.40      0.43       139
           2       0.07      0.13      0.09        23
           3       0.10      0.16      0.12        19
           8       0.87      0.44      0.58       264
           9       0.91      0.88      0.90       145
          11       0.57      0.76      0.65        21
          12       0.96      0.72      0.82       431
          13       0.21      0.43      0.29         7
          14       0.00      0.00      0.00        22
          16       0.92      0.73      0.82       485
          18       0.00      0.00      0.00         5
          19       0.45      0.07      0.12       702
          20       0.73      0.81      0.77     16112
          21       0.00      0.00      0.00        10
          

c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\ADMIN\OneDrive\Máy tính\TTCS\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape